In [ ]:
# set library
import json
from transformers import AutoTokenizer
from tqdm import tqdm
import re
import numpy as np
from collections import defaultdict
from scipy import integrate
import scipy
import math

In [ ]:
# Set Variables
model_flops = {
    'meta-llama/Llama-3.2-3B-Instruct': 3000000000,
    'Qwen/Qwen2.5-3B-Instruct': 3000000000,
    'google/gemma-3-4b-it': 4000000000,
    'Qwen/Qwen2.5-7B-Instruct': 7000000000,
    'google/gemma-3-27b-it': 27000000000,
}

# dataset="gsm8k"
dataset = "math"
# model = 'meta-llama/Llama-3.2-3B-Instruct'
model = "Qwen/Qwen2.5-7B-Instruct"
# model = 'google/gemma-3-4b-it'

cp_threshold = 0.90
beta_threshold = 0.95

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)
if dataset == 'omnimath':
    with open(f"./logs/self_certainty/sc_16_{dataset}_2048_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
else:
    with open(f"./logs/self_certainty/sc_16_{dataset}_1024_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

In [ ]:
def apply_chat_template(dataset, question, response):
    if dataset == "gpqa_diamond" or dataset == "arcChallenge":
        prompt = f"{question}\n\nBased on the above, what is the single, most likely answer choice? Answer in the format \"The correct answer is (insert answer here)\"."
        chat_template = [{'role': 'user', 'content': prompt}, {"role": "assistant", "content": response}]
    else:
        chat_template = [{'role': 'user', 'content': question}, {"role": "assistant", "content": response}]

    return tokenizer.apply_chat_template(chat_template, tokenize=False, add_generation_prompt=False)

def extract_boxed_content(text):
    start = text.find(r"\boxed{")
    if start == -1:
        return None
    i = start + len(r"\boxed{")
    depth = 1
    content = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        content.append(c)
        i += 1
    return "".join(content)

def clean(value):
    if dataset == 'gsm8k':
        value = value.replace(',','')
        numbers = re.findall(r"\d+(?:\.\d+)?", value)
        if len(numbers) > 0:
            value = numbers[-1]
        else:
            value= None
    else:
        final_value = extract_boxed_content(value)
        if final_value is None:
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = re.findall(r"A|B|C|D|a|b|c|d", value)
                value = value[0].lower() if value else ''
            else:
                value = ""
        else:
            value = final_value.replace(' ','')
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = value.lower()
    return value

In [ ]:
# aggregate results by question
results = defaultdict(list)
error_cnt=0
for inst in data:
    question = inst['question']


    if dataset == 'gsm8k':
        pred = inst['pred'].replace(' ','').strip()
        verdict = clean(inst['pred']) == clean(inst['answer'])
    else:
        pred = clean(inst['response'])
        if pred == '':
            pred = clean(inst['pred'])
            if pred == '':
                pred = inst['pred'].replace(' ','').strip()
        
        if pred == '':
            error_cnt += 1
        
        verdict = inst['verdict']

    response = inst['response']
    if pred == '':
        error_cnt+=1
    internal_value = None
    
    if question not in results:
        results[question] = []
    
    if dataset == 'gsm8k':
        results[question].append((clean(pred), internal_value, verdict, apply_chat_template(dataset, question, response)))
    else:
        results[question].append((pred, internal_value, verdict, apply_chat_template(dataset, question, response)))

# SC results

In [ ]:
majority_voting_results = defaultdict(list)
for question in results:
    majority_voting_results[question] = []
    for i in range(16):
        answer_dict={}
        answer_correct={}
        i = i+1
        for inst in results[question][:i]:
            answer , verdict = inst[0], inst[2]
            if answer not in answer_dict:
                answer_dict[answer] = 0
            answer_dict[answer] += 1
            if answer not in answer_correct:
                answer_correct[answer] = {}
            answer_correct[answer][verdict] = answer_correct[answer].get(verdict, 0) + 1

        
        final_pred = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)[0][0]
        sorted_answer_dict = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)

        if len(sorted_answer_dict) == 1:
            a, b = sorted_answer_dict[0][1], 0
        else:
            a, b = sorted_answer_dict[0][1], sorted_answer_dict[1][1]
        a = float(a)
        b = float(b)
        
        prob = integrate.quad(lambda x : x**(a) * (1-x)**(b), 0.5, 1)[0] / integrate.quad(lambda x : x**(a) * (1-x)**(b), 0, 1)[0]
        if dataset == 'arcChallenge' or dataset == 'gpqa_diamond':
            majority_votig_results[question].append((final_pred.lower() == question_answer[question].lower(), prob))
        else:
            majority_voting_results[question].append((answer_correct[final_pred].get(True, 0) >= answer_correct[final_pred].get(False, 0), prob))

In [ ]:
sc_results=[]
sample_size_list=[]
cnt=0
total_length = 0
for question in majority_voting_results:
    for i, inst in enumerate(majority_voting_results[question]):
        correctness, prob = inst
        response = results[question][i][3]
        total_length += len(tokenizer.encode(response, add_special_tokens=False))
    sample_size_list.append(len(majority_voting_results[question]))
    cnt += (majority_voting_results[question][-1][0] == True)
    sc_results.append(majority_voting_results[question][-1][0] == True)

print(f"====================SC (k=16)====================")
print("SC Results:")
print("Correct:", cnt)
print("Total:", len(majority_voting_results))
print("Accuracy: {:.2f}%".format(cnt/len(majority_voting_results) * 100))
print("Average Sample Size:", np.mean(sample_size_list))
print("Average Response Length per Question:", total_length/len(majority_voting_results))
print("Total Response Length:", total_length)
print("Flops: ", total_length * model_flops[model], "FLOPS")
print("Average TFLOPS:", (total_length * model_flops[model]) / (len(sample_size_list) * 1e12))
print("==========================================")